#### 기본 설정

In [1]:
!pip install -r requirements.txt

  Using cached torch-2.8.0-cp312-cp312-win_amd64.whl.metadata (30 kB)
Using cached torch-2.8.0-cp312-cp312-win_amd64.whl (241.3 MB)


In [2]:
import os, json, time, random
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
sys.path.append("./src") 

import TokenSample
from RecommendAgent import RecommendationAgent
from save_result import save_recommendations_to_csv
from Gemini import safe_gemini_call, llm_judge
from eval_metrics import save_metrics

c:\anaconda3\envs\qwen\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- 사용자 본인의 값으로 모두 변경 필수

In [3]:
#사용자 본인의 값으로 모두 변경 필수 (아래는 환경변수에서 로드, 하드코딩 금지)

MAC_ADDRESS = os.environ["KISTI_MAC_ADDRESS"]
KISTI_CLIENT_ID = os.environ["KISTI_CLIENT_ID"]
KISTI_KEY = os.environ["KISTI_KEY"]

DATAON_SEARCH_KEY = os.environ["DATAON_SEARCH_KEY"]
DATAON_DETAIL_KEY = os.environ["DATAON_DETAIL_KEY"]

- ScienceOn의 Token은 2시간마다 한 번씩 갱신 필수

In [4]:
#ScienceOn의 Token은 2시간마다 한번씩 갱신 필수

# 새 토큰 발급
tokens = TokenSample.createToken(
    mac_address=MAC_ADDRESS,
    client_id=KISTI_CLIENT_ID,
    key_value=KISTI_KEY
)

# tokens 발급 확인
print(tokens)

KISTI_REFRESH_TOKEN  = TokenSample.KISTI_REFRESH_TOKEN
KISTI_ACCESS_TOKEN = TokenSample.KISTI_ACCESS_TOKEN

In [5]:
QWEN_ENDPOINT = "https://aida.kisti.re.kr:10414/v1/chat/completions"
QWEN_TOKEN_PATH = r".\src\qwen3_14b_token.txt"
LLM_CALL_DELAY = 1  # 초당 호출 간격

try:
    with open(QWEN_TOKEN_PATH, "r", encoding="utf-8") as f:
        QWEN_TOKEN = f.read().strip()
except FileNotFoundError:
    print(f"⚠ Qwen 토큰 파일을 찾을 수 없습니다: {QWEN_TOKEN_PATH}")
    QWEN_TOKEN = None

In [6]:
agent = RecommendationAgent(
    {'search_key': DATAON_SEARCH_KEY, 'detail_key': DATAON_DETAIL_KEY},
    {'client_id': KISTI_CLIENT_ID, 'refresh_token': KISTI_REFRESH_TOKEN},
    {'endpoint': QWEN_ENDPOINT, 'token': QWEN_TOKEN}
)


In [7]:

dsid = input("dataset_id 입력: ").strip()
if not dsid:
    raise SystemExit("dataset_id 미입력, 종료합니다.")

results = agent.recommend(dsid, top_k_final=5)



[INFO] '의료영상데이터' 기반 추천 시작...
 - 생성된 키워드: ['Medical Imaging', 'Digital Twin', 'Personalized Medicine', 'AI in Healthcare', 'Healthcare Data Analysis']


In [8]:
print("\n==================== 최종 추천 결과 확인 ====================")
for i, r in enumerate(results, 1):
    print(f"\n[{i}] {r['구분']}")
    print(f"\n[{i}] {r['title']}")
    print(f" - 설명: {r.get('desc','')[:200]}...")
    print(f" - 점수: {r['score']} ({r['level']})")
    print(f" - 추천 사유: {r['추천 사유']}")
    print(f" - URL: {r['url']}")
print("\n========================================================")

#최종 추천 결과 저장
save_recommendations_to_csv(results)


==================== 최종 추천 결과 확인 ====================

[1] paper

[1] 초고속 정보통신망을 통한 3차원 영상 정보의 가상현실 관리에 관한 연구
 - 설명: 본 논문에서는 각종 단층 촬영 의료영상 장비로 촬영한 2차원 단면화상 데이터들을 차원 재구성 알고리즘을 사용하여 3차원 영상으로 재구성한 다음, 웹 서버의 데이터베이스에 저장하고 관리하며, 인터넷 가상현실 표준언어인 VRML(Virtual Reality Modeling Language)로 표현된 3차원 의료영상을 비롯한 각종 의료영상 정보를 웹브라우저를 사...
 - 점수: 66.29 (추천)
 - 추천 사유: 두 데이터는 **3차원 영상 정보의 처리 및 활용**이라는 핵심적인 공통점을 가진다. 추천 데이터는 입력 데이터인 의료영상데이터를 기반으로, 초고속 정보통신망을 통해 3차원 영상 정보를 가상현실 환경에서 관리하는 기술적 접근을 제시함으로써, 의료영상의 실시간 처리, 고해상도 시각화, 멀티유저 협업 등 기존 의료영상데이터에서 다루지 못했던 **네트워크 기반의 가상현실 통합 관리 시스템**을 구체화한다. 이는 의료영상의 활용 범위를 확장하고, 진단 정확도 향상 및 원격 의료 협업 가능성을 새로운 차원으로 끌어올리는 데 기여한다.
 - URL: https://scienceon.kisti.re.kr/srch/selectPORSrchArticle.do?cn=JAKO199811919549453

[2] paper

[2] 딥러닝 기반 의료영상 분석을 위한 데이터 증강 기법
 - 설명: 영상처리 기반으로 의료영상을 분석하는 기법은 정상 환자와 비정상 환자를 분류, 병변 검출 및 장기나 병변의 분할 등에 사용되고 있다. 최근 인공지능 기술의 비약적 발전으로 의료영상 분석 연구들이 딥러닝 기술을 활용하여 시도되고 있다. 의료영상은 학습에 필요한 데이터를 충분히 모으기 어렵고 클래스별 데이터 수의 차이 때문에, 딥러닝 모델의 성능을 올리는데 어...
 - 점수: 

'./outputs\\recommend_result.csv'

In [9]:
DATAON_PATH = "./data/dataon_satellite.json"
OUT_DIR = "./data"
os.makedirs(OUT_DIR, exist_ok=True)

RECOMM_CKPT = os.path.join(OUT_DIR, "qwen_recommend_checkpoint.json")
OUT_FILE = os.path.join(OUT_DIR, "qwen_recommend_results.json")
LLM_CALL_DELAY = 1  # 초당 호출 간격

In [10]:
with open(DATAON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
if isinstance(data, dict) and "records" in data:
    data = data["records"]
    
random.seed(42)
all_ids = [(d.get("raw") or {}).get("svc_id") for d in data]
random.shuffle(all_ids)

In [ ]:
recommendations = []
if os.path.exists(RECOMM_CKPT):
    try:
        recommendations = json.load(open(RECOMM_CKPT, "r", encoding="utf-8"))
        tqdm.write(f"[INFO] 체크포인트 복원됨: {len(recommendations)}개")
    except:
        recommendations = []

done_ids = {r["svc_id"] for r in recommendations}

# --- 50개 루프 ---
for dataset_id in tqdm(all_ids, desc="추천 생성 중", ncols=100):
    if dataset_id in done_ids:
        continue
    try:
        recs = agent.recommend(dataset_id, top_k_final=5)
        if recs:
            title = agent.dataon.detail(dataset_id).get("title", "")
            recommendations.append({
                "svc_id": dataset_id,
                "title": title,
                "recommendations": recs
            })
            done_ids.add(dataset_id)
            tqdm.write(f"[OK] 저장: {dataset_id} (현재 {len(recommendations)}/50)")
    except Exception as e:
        tqdm.write(f"[WARN] {dataset_id} 처리 중 오류: {e}")

    # --- 체크포인트 저장 ---
    tmp_ckpt = RECOMM_CKPT + ".tmp"
    json.dump(recommendations, open(tmp_ckpt, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    os.replace(tmp_ckpt, RECOMM_CKPT)
    if len(recommendations) % 10 == 0:
        tqdm.write(f"[CHECKPOINT] {len(recommendations)} 저장 완료.")
    if len(recommendations) >= 50:
        break
    time.sleep(5)  

# --- 최종 저장 ---
json.dump(recommendations, open(OUT_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
tqdm.write(f"\n 최종 추천 결과 50개 저장 완료: {OUT_FILE}")

[INFO] 체크포인트 복원됨: 6개


추천 생성 중:   0%|                                                        | 0/7967 [00:00<?, ?it/s]


[INFO] 'Attività POLIZIA GIUDIZIARIA anno 2017' 기반 추천 시작...
 - 생성된 키워드: ['Open Data', 'Data Visualization', 'Data Portals', 'Metadata Standards', 'User Experience Design']


추천 생성 중:   0%|                                                        | 0/7967 [01:41<?, ?it/s]     

[OK] 저장: 9b1002bed1caad912ac7244948355259 (현재 7/50)


...


[INFO] 'Benutzerbewertung Unterkunft Sonderservice SoL - Der Benutzer gedeiht nie zu Hause, Prozentsatz (%)' 기반 추천 시작...
 - 생성된 키워드: ['Social Housing Evaluation', 'User Experience in Residential Services', 'Specialized Housing Effectiveness', 'Residential Service Quality Assessment', 'Sense of Belonging in Housing']


추천 생성 중:   0%|                                           | 1/7967 [03:39<235:18:25, 106.34s/it]     

[OK] 저장: d08a230758e777e701d19ae4f79800a4 (현재 8/50)


추천 생성 중:   0%|                                           | 2/7967 [03:44<250:47:07, 113.35s/it]


[INFO] 'Species and sex specific chemical composition in a concealed gland from an internal gland-like tissue of an African frog family' 기반 추천 시작...
 - 생성된 키워드: ['Amphibian chemical ecology', 'Sexual dimorphism in amphibians', 'Chemical defense mechanisms', 'Amphibian gland secretions', 'Comparative physiology of amphibians']


In [ ]:
RECOMM_FILE = "./data/qwen_recommend_results.json"
OUT_DIR = "./data/qwen_satellite_pseudo_gt"
OUT_FILE = os.path.join(OUT_DIR, "qwen_llm_judge_results.json")
CKPT_FILE = os.path.join(OUT_DIR, "qwen_judge_checkpoint.json")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Gemini API 키는 하드코딩하지 않고 환경변수(콤마 구분)에서 로드
GEMINI_KEYS = [k for k in os.environ.get("GEMINI_API_KEYS", "").split(",") if k]
if not GEMINI_KEYS:
    raise RuntimeError("환경변수 GEMINI_API_KEYS가 설정되어 있지 않습니다.")
KEY_INDEX = 0
LLM_DELAY = 0.2

In [ ]:
with open(RECOMM_FILE, "r", encoding="utf-8") as f:
    rec_data = json.load(f)

judged = []

if os.path.exists(CKPT_FILE):
    try:
        judged = json.load(open(CKPT_FILE, "r", encoding="utf-8"))
        print(f"[INFO] 체크포인트 복원됨: {len(judged)}개")
    except:
        judged = []

done_ids = {j["svc_id"] for j in judged}
todo = [r for r in rec_data if r["svc_id"] not in done_ids]

for rec in tqdm(todo, desc="Pseudo-Judge 평가 중", ncols=100):
    q_title = rec["title"]
    results = []
    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(llm_judge, q_title, c["title"], c["추천 사유"]): c for c in rec["recommendations"]}
        for fut in as_completed(futures):
            c = futures[fut]
            try:
                score, reason = fut.result()
            except Exception as e:
                score, reason = 0.0, f"(error) {e}"
            c["judge_score"] = score
            c["judge_reason"] = reason
            results.append(c)
            time.sleep(LLM_DELAY)

    judged.append({
        "svc_id": rec["svc_id"],
        "title": q_title,
        "evaluated": results
    })

    json.dump(judged, open(CKPT_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"[CHECKPOINT] {len(judged)}개 평가 저장 중...")

# =========================================================
# 최종 저장
# =========================================================
json.dump(judged, open(OUT_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\n 최종 평가 결과 저장 완료: {OUT_FILE}")


In [ ]:
JUDGE_FILE =r".\data\qwen_satellite_pseudo_gt\qwen_llm_judge_results.json" 

In [ ]:
metrics = save_metrics(JUDGE_FILE,  k=5, threshold=2.0)
print(metrics)